<a href="https://colab.research.google.com/github/veneela-2811/Story_Telling_App_Recommendation_Engine/blob/main/Story.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Import necessary libraries

In [ ]:
import numpy as np
import pandas as pd

#Import the datasets

In [ ]:
df=pd.read_csv('Story_Synthetic_dataset.csv')

In [ ]:
df.head()

# Popularity Based Recommender System

##recommendation of stories based on the number of ratings and average rating. The basic idea is that stories with more ratings or higher average ratings are recommended to users.

###Finding number of ratings for each book

In [ ]:
num_rating_df = df.groupby('story_id').count()['rating'].reset_index()
num_rating_df.rename(columns={'rating':'num_ratings'},inplace=True)
num_rating_df

###Finding average rating for each story

In [ ]:
avg_rating_df = df.groupby('story_id')['rating'].mean().reset_index()
avg_rating_df.rename(columns={'rating':'avg_rating'},inplace=True)
avg_rating_df

###Popularity based on average rating

In [ ]:
popular_df = num_rating_df.merge(avg_rating_df,on='story_id')
popular_df.head()

In [ ]:
# Sort by number of ratings and average rating
top_5_stories = popular_df.sort_values(['avg_rating','num_ratings'], ascending=False).head(5)

# Display the top 5 popular stories
print(top_5_stories)



###Filter Active Users – Select users who have rated at least 5 stories.

In [ ]:
x = df.groupby('user_id').count()['rating'] >= 5
qualified_users = x[x].index

###Filter Popular Books – Keep books with at least 5 ratings.

Refine Ratings Dataset – Include only ratings from selected users for popular books.

In [ ]:
filtered_rating = df[df['user_id'].isin(qualified_users)]

In [ ]:
y = filtered_rating.groupby('story_id').count()['rating']>=5
famous_books = y[y].index

In [ ]:
final_ratings = filtered_rating[filtered_rating['story_id'].isin(famous_books)]

In [ ]:
final_ratings

###Create User-Item Matrix – Convert data into a pivot table where:Rows → Story_ID's,Columns → USER IDs,Values → Ratings

In [ ]:
pt = final_ratings.pivot_table(index='story_id',columns='user_id',values='rating')

In [ ]:
pt.fillna(0,inplace=True)
pt

#Compute Similarity Scores – Use cosine similarity on the user-item matrix to find story similarities.
###Find Similar Books – Sort stories based on similarity scores and pick the top recommendations.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
similarity_scores = cosine_similarity(pt)
similarity_scores

# Collaborative Filtering Based Recommender System

In [ ]:
def recommend(story_id):
    #Find the index of the story in the pivot table
    if story_id not in pt.index:
        return f"story ID {story_id} not found in the filtered data."

    index = pt.index.get_loc(story_id)

    # Get top 2 most similar stories
    similar_items = sorted(
        list(enumerate(similarity_scores[index])),
        key=lambda x: x[1],
        reverse=True
    )[1:3]  # skip the input story

    data = []
    for i in similar_items:
        similar_story_id = pt.index[i[0]]
        temp_df = df[df['story_id'] == similar_story_id].drop_duplicates('story_id')

        if not temp_df.empty:
            item = [
                temp_df['title'].values[0]
            ]
            data.append(item)

    return data

In [ ]:
recommend(2)

#Time based sorting

In [ ]:
from datetime import datetime

# Merge publication_year into popular_df from the original df
publication_year_df = df[['story_id', 'publication_year']].drop_duplicates()
popular_df = popular_df.merge(publication_year_df, on='story_id', how='left')

#Calculate years since publication
current_year = datetime.now().year
popular_df['years_since_publication'] = current_year - popular_df['publication_year']
popular_df['years_since_publication'] = popular_df['years_since_publication'].replace(0, 1)

#Calculate time-based popularity
popular_df['time_based_popularity'] = popular_df['num_ratings'] / popular_df['years_since_publication']

#Sort by time-based popularity
sorted_books = popular_df.sort_values(by='time_based_popularity', ascending=False)

#Display top 10 popular stories
print(sorted_books[['story_id', 'num_ratings', 'years_since_publication', 'time_based_popularity']].head(10))


#Time period based recommendation


In [ ]:
import pandas as pd


df['rating_year'] = df['timestamp']

#Compute yearly trends per story
yearly_trends = df.groupby(['rating_year', 'story_id']).agg(
    rating_count=('rating', 'count'),
    avg_rating=('rating', 'mean')
).reset_index()

#Merge with story titles and publication year
story_info = df[['story_id', 'title', 'publication_year']].drop_duplicates()
yearly_trends = yearly_trends.merge(story_info, on='story_id', how='left')

#Function to get trending stories in a year range
def get_trending_stories(start_year, end_year):
    filtered = yearly_trends[
        (yearly_trends['rating_year'] >= start_year) &
        (yearly_trends['rating_year'] <= end_year)
    ]

    # Aggregating total ratings and average ratings
    popular_stories = filtered.groupby(['story_id', 'title', 'publication_year']).agg(
        total_ratings=('rating_count', 'sum'),
        overall_avg_rating=('avg_rating', 'mean')
    ).reset_index()

    # Sort
    popular_stories = popular_stories.sort_values(
        by=['total_ratings', 'overall_avg_rating'], ascending=[False, False]
    )

    return popular_stories

# # Example: Get top stories from 2019 to 2022
# start_year = 2019
# end_year = 2022
# popular_stories = get_trending_stories(start_year, end_year)

# # 6. Show top 10
# print(f"Top Trending Stories from {start_year} to {end_year}:\n")
# print(popular_stories.head(5))


#Context-aware recommendation.

##Using Calenderific API to fetch relevant stories based on occasions in a particular month

In [ ]:
!pip install PyMuPDF

In [ ]:
def list_extracted_files(extract_path):
    files = os.listdir(extract_path)
    print(f"[INFO] Extracted Files: {files}")

##If month or occasion specified,gives relevent festivals accordingly.Else,gives stories related to occasions according to the current month.

In [ ]:
import zipfile
import os
from datetime import datetime
import requests
import unicodedata
from sentence_transformers import SentenceTransformer, util

#Model Loading
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

#Normalize text
def normalize_text(text):
    text = text.lower().strip()
    return unicodedata.normalize("NFC", text)

#Occasion Keywords
occasion_keywords = {
    "Diwali": [
        "దీపాలు", "పండుగ", "అలంకారాలు", "తీపులు", "కుటుంబం", "ఆనందం",
        "దీపాల వరుస", "పటాకులు", "లడ్డూ", "నూతన దుస్తులు", "కృష్ణుడు",
        "లాంపులు", "దీపోత్సవం", "పూజ", "కాంతులు"
    ],
    "Children's Day": [
        "పిల్లలు", "బాల్యం", "ఆటలు", "పాఠశాల", "సంతోషం", "శిక్షణ",
        "జవహర్‌లాల్ నెహ్రూ", "టీచర్లు", "ఉత్సవాలు", "పోటీలు", "బహుమతులు",
        "నాటికలు", "కథలు", "బొమ్మలు", "కలల ప్రపంచం"
    ],
    "Christmas": [
        "క్రిస్మస్", "బహుమతులు", "చిన్నారి", "చర్చి", "సంతోషం", "తల్లి తండ్రులు",
        "క్రిస్మస్ చెట్టు", "సాంటా క్లాజ్", "జింగిల్ బెల్స్", "క్రిస్మస్ పాటలు",
        "గుడ్లు", "క్రిస్మస్ స్టార్", "క్రిస్మస్ పిండి వంటలు", "ప్రార్థనలు", "మౌలికత్వం"
    ],
    "Dussehra": [
        "దసరా", "విజయదశమి", "రావణ దహనం", "అయోధ్య", "రాముడు", "సీత",
        "హనుమాన్", "రామాయణం", "పూజ", "బొమ్మల కోలువు", "శక్తి పూజ",
        "దుర్గమ్మ", "ఆలయం", "నవరాత్రులు", "ఆనందం"
    ],
    "Independence Day": [
        "స్వాతంత్ర్య దినోత్సవం", "జెండా", "పతాకావందనం", "భారతదేశం", "జవాన్లు",
        "ఆజాదీ", "గణతంత్రం", "ప్రముఖ నాయకులు", "సభలు", "రాష్ట్ర గీతం",
        "పరేడ్", "పరాక్రమం", "దేశభక్తి", "పాత్రత", "మంచి పౌరుడు"
    ],
    "Republic Day": [
        "గణతంత్ర దినోత్సవం", "భారత రాజ్యాంగం", "డాక్టర్ అంబేద్కర్", "జెండా ఊపడం",
        "రాజ్ పథ్ పరేడ్", "సైనిక ప్రదర్శన", "జాతీయ గీతం", "త్రివర్ణ పతాకం",
        "పతాకావందనం", "సాంస్కృతిక ప్రదర్శనలు", "ప్రముఖ అతిథులు", "భవిష్యత్ భావనలు",
        "దేశభక్తి పాటలు", "ప్రమాణ స్వీకారం", "భారతీయత"
    ],
    "Friendship Day": [
        "మిత్రత్వం", "స్నేహితులు", "స్నేహం", "పండుగ", "బంధం", "ఆనందం",
        "బహుమతులు", "స్నేహపత్రికలు", "జ్ఞాపకాలు", "సంబంధాలు", "ఆప్యాయత",
        "స్నేహసూక్తులు", "పాటలు", "పిక్నిక్", "సెల్ఫీలు",
        "సహచరులు", "పరిచయాలు", "మాటలు", "సమ్మేళనం", "ఆరాధనలు"
    ],
    "Raksha Bandhan": [
        "రాఖీ", "బంధం", "సోదరుడు", "సోదరి", "సంకల్పం", "సురక్షితుడు",
        "బంధు", "పండుగ", "సహాయం", "ఆప్యాయత", "స్నేహం",
        "సంధి", "బంధువు", "కటాక్షం", "పరిశుభ్రత", "బంధం ప్రతిజ్ఞ",
        "తల్లి", "దుప్పటి", "పండుగ వాతావరణం", "ఆచారాలు"
    ]
}

occasion_keywords = {
    normalize_text(k): [normalize_text(w) for w in v] for k, v in occasion_keywords.items()
}

#Extract ZIP of Text Files
def extract_zip(zip_path, extract_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)


# Read Text Files and Load Stories
def load_stories_from_text_files(folder_path):
    stories = []
    for foldername, subfolders, filenames in os.walk(folder_path):  # Traverse all subfolders
        for filename in filenames:
            if filename.endswith(".txt"):
                file_path = os.path.join(foldername, filename)
                try:
                    with open(file_path, 'r', encoding='utf-8') as file:
                        content = file.read()
                        if content.strip():
                            stories.append({
                                "title": filename.replace(".txt", ""),
                                "content": content.strip()
                            })
                except Exception as e:
                    print(f"[ERROR] Failed to read {filename}: {e}")
    return stories


#Fetch Holidays using Calendarific API
def get_festivals_from_calendarific(api_key, country="IN", month=None, year=None):
    if not month:
        month = datetime.now().month
    if not year:
        year = datetime.now().year

    try:
        res = requests.get("https://calendarific.com/api/v2/holidays", params={
            "api_key": api_key,
            "country": country,
            "year": year,
            "month": month
        })
        res.raise_for_status()
        holidays = res.json()["response"]["holidays"]
        return [normalize_text(h["name"]) for h in holidays]
    except Exception as e:
        print(f"[ERROR] Failed to fetch holidays: {e}")
        return []

#Recommend Stories
def recommend_stories_with_api(stories, occasion_keywords, api_key, user_selected_occasion=None, month=None, year=None, top_n=2):
    #Determine Occasion
    occasion = None
    if user_selected_occasion:
        occasion = normalize_text(user_selected_occasion)
    else:
        fetched_festivals = get_festivals_from_calendarific(api_key, month=month, year=year)
        print("[INFO] Fetched Festivals from API:", fetched_festivals)
        for fest in fetched_festivals:
            if fest in occasion_keywords:
                occasion = fest
                break

    if not occasion:
        print("[INFO] No matching occasion found.")
        return []

    if occasion not in occasion_keywords:
        print(f"[WARNING] No keywords defined for occasion: {occasion}")
        return []

    #Embed keywords
    keywords_text = " ".join(occasion_keywords[occasion])
    keywords_embedding = model.encode(keywords_text, convert_to_tensor=True)

    #Score stories
    story_scores = []
    for story in stories:
        story_embedding = model.encode(story["content"], convert_to_tensor=True)
        similarity = util.pytorch_cos_sim(story_embedding, keywords_embedding).item()
        story_scores.append((story["title"], similarity))

    #Display results
    sorted_stories = sorted(story_scores, key=lambda x: x[1], reverse=True)
    top_stories = sorted_stories[:top_n]

    print(f"\n[INFO] Occasion detected: {occasion.title()}")
    print(f"Top {top_n} relevant stories:\n")
    for i, (title, score) in enumerate(top_stories, start=1):
        print(f"{i}. {title} (Similarity: {score:.2f})")

    return top_stories

# === Main Execution ===

#Define paths
zip_path = "/content/Telugu_stories_text_files (2).zip"
extract_path = "/content/telugu_stories_extracted"

def list_extracted_files(extract_path):
    files = os.listdir(extract_path)
    print(f"[INFO] Extracted Files: {files}")

#Extract ZIP
extract_zip(zip_path, extract_path)
list_extracted_files(extract_path)

#Load stories from Text Files
stories = load_stories_from_text_files(extract_path)
print(f"[INFO] Loaded {len(stories)} stories.")

#Recommend based on detected or selected occasion
api_key = "dP1ErMp4DJVNNUdOC1yFo8Y8lG31dWKl"
recommend_stories_with_api(stories, occasion_keywords, api_key, user_selected_occasion=None, month=8)
